# Samples from models on 60km -> 2.2km-4x over Birmingham

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import IPython
import matplotlib
import matplotlib.pyplot as plt

from mlde_analysis.furflex_data import prep_eval_data
from mlde_analysis.examples import plot_examples, em_timestamps
from mlde_analysis import cp_model_rotated_pole, STYLES

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, CPM_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

In [ ]:
examples_to_plot = {
    "CPM": {
        "band": "2073-02-17",
    },
}   

In [ ]:
import hvplot.xarray

In [ ]:
for source, examples in examples_to_plot.items():
    IPython.display.display_html(f"<h2>{source} Samples</h2>", raw=True)
    # fig_height = 4.5
    # fig = plt.figure(layout="constrained", figsize=(fig_width, fig_height))
    for label, date in examples.items():
        example_ds = EVAL_DS[source].isel(time=slice(24)).isel(ensemble_member=0)
        example_ds["target_pr"].isel(time=0).hvplot()


In [ ]:
def hook(plot, element):
    # print('plot.state:   ', plot.state)
    # print(type(plot.handles["plot"]))
    # print('plot.handles: ', sorted(plot.handles.keys()))
    # plot.handles['xaxis'].axis_label_text_color = 'red'
    # plot.handles['yaxis'].axis_label_text_color = 'blue'
    plot.handles['plot'].coastlines(cp_model_rotated_pole, scale="10m")
    # fig = plot.handles["plot"].set_norm()
import xarray as xr
import pandas as pd
import numpy as np



def bin_data_with_boundaries(da, boundaries):
    """Bin multidimensional data using boundaries (works like BoundaryNorm)."""
    bin_labels = [f"{boundaries[i]:.2g}-{boundaries[i+1]:.2g}" 
                  for i in range(len(boundaries)-1)]
    
    def _bin(x):
        # np.digitize returns indices 0, 1, 2, ... for each boundary bin
        # Subtract 1 because digitize returns 1-indexed
        indices = np.digitize(x, boundaries) - 1
        # Clip to valid range (handles values < min or > max)
        indices = np.clip(indices, 0, len(bin_labels) - 1)
        # Map indices to labels
        return np.array(bin_labels)[indices]
    
    return xr.apply_ufunc(
        _bin, 
        da, 
        vectorize=False,  # Let it handle the full array
        dask='allowed'
    )

# Usage
precip_levels = [
    0.1,
    0.5,  # 1,
    1,  # 2.5,
    2,  # 5,
    3,  # 7.5,
    4,  # 10,
    5,  # 15,
    6,  # 20,
    7,  # 30,
    8,  # 40,
    10,  # 50,
    20,  # 70,
    30,  # 100,
    50,  # 150,
]
precip_binned = bin_data_with_boundaries(EVAL_DS["CPM"]["target_pr"].isel(time=200), precip_levels)


# precip_binned.hvplot(
#     kind="quadmesh", coastline=True, projection=cp_model_rotated_pole,
# )#.opts(hooks=[hook])

# precip_binned.hvplot.quadmesh(
#     # x='grid_longitude', y='grid_latitude',
#     projection=cp_model_rotated_pole,
#     cmap=STYLES["pr"]["cmap"],
#     coastline=True
# )

from bokeh.models import LinearColorMapper

base_colors=STYLES["pr"]["cmap"]
low = 0
high = 80
bound_colors = []
j = 0
for i in range(0, high*20, 1):
    pr = i/20
    bound_colors.append(matplotlib.colors.to_hex(STYLES["pr_hourly"]["cmap"](STYLES["pr_hourly"]["cnorm"](pr))))
# color_mapper = LinearColorMapper(palette=bound_colors, low=low, high=80, nan_color="white")

EVAL_DS["CPM"]["target_pr"].isel(time=slice(200, 224)).hvplot.image(
    x="grid_longitude", y="grid_latitude",
    cmap=bound_colors,#STYLES["pr"]["cmap"],
    # color_mapper=color_mapper,
    coastline="10m",
    # color_levels=precip_levels,
    clim=(0,high),
    dynamic=True,
    projection=cp_model_rotated_pole,
    crs=cp_model_rotated_pole,
)

In [ ]:
STYLES["pr"]["cmap"](STYLES["pr"]["norm"](17))

In [ ]:
data_binned

In [ ]:
EVAL_DS[source]["target_pr"].argmax()

In [ ]:
152529 % (1*64*64)

In [ ]:
EVAL_DS["CPM"]["target_pr"]

In [ ]:
EVAL_DS["CPM"]["target_pr"].isel(ensemble_member=0, time=0).drop_vars(["year", "yyyymmddhh", "time_period", "dec_adjusted_year", "stratum", "tp_season_year", "month_number", "latitude", "longitude"]).hvplot.explorer(y="grid_latitude", x="grid_longitude")

In [ ]:
example_ds

In [ ]:
for source, examples in examples_to_plot.items():
    IPython.display.display_html(f"<h2>{source} Samples</h2>", raw=True)

    if source == "CPM":
        fig_width = 6
    else:
        fig_width = 4
    fig_height = 4.5
    fig = plt.figure(layout="constrained", figsize=(fig_width, fig_height))
    plot_examples(
        EVAL_DS[source], examples,
        vars=eval_vars, models=MODELS[source], fig=fig, sim_title=source, examples_sample_idxs=examples_sample_idxs, inputs=example_inputs,
    )
    plt.show()